In [0]:
from pyspark.sql import functions as F

# Drop duplicated records: from BatchRead 
from pyspark.sql import functions as F
json_schema = f"""
    order_id STRING,
    order_timestamp TIMESTAMP,
    customer_id STRING,
    quantity BIGINT,
    total BIGINT,
    books ARRAY<STRUCT<
        book_id: STRING,
        quantity: LONG,
        subtotal: DOUBLE>>
"""


def upsert_data(microBatchDF, batch):
    microBatchDF.createOrReplaceTempView('orders_microbatch')

    sql_query = """
        MERGE INTO dev.silver.orders_silver o
        USING orders_microbatch b
        ON o.order_id = b.order_id
        WHEN NOT MATCHED THEN INSERT *
    """

    microBatchDF.sparkSession.sql(sql_query) # For clusters with runtime 11.3 or above 
    #microBatchDF._jdf.sparkSession().sql(sql_query) # For clusters with runtime 10.5 or below

def process_orders_silver():
    dedup_stream = (
        spark.readStream.table('dev.bookstore_bronze.bookstore_bronze')
            .filter(F.col("topic") == 'orders')
            .select(
                F.from_json(
                    F.col('value').cast('string'),
                    schema= json_schema
                ).alias('v')        
            ).select('v.*')
            .withWatermark('order_timestamp', "30 seconds")
            .dropDuplicates(['order_id', 'order_timestamp'])
    )

    df = (
        dedup_stream
            .filter(F.col("quantity") > 0)
            .writeStream
            .foreachBatch(upsert_data)
            .option("checkpointLocation", "/Volumes/dev/landing_zone/kafka_source/checkpoints/silver_orders/")
            .trigger(availableNow=True)
            .start()
    )
    df.awaitTermination()


process_orders_silver()